# resnet_finetune_300

Generated with **Lispyn** — originally created by **Eduardo J. Barrios**.

Source: [Lispyn](https://github.com/edujbarrios/lispyn) (Apache-2.0).

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='resnet_finetune_300', assignments=[['md', 'md']], prefix='')


# ResNet18 — transfer learning + full fine-tuning

An end-to-end **PyTorch** experiment written in Lispyn. It loads labeled images, splits by class, runs head-only tuning, then unfreezes the backbone for **300 epochs**, and produces publication-ready curves.

**Author:** Eduardo J. Barrios · **Source:** Lispyn.

In [ ]:
import hy
import torch
import torchvision
from lispyn.media import image_folder
from lispyn.data import split_dataset
from lispyn.reproducibility import seed_all
seed_all(42)
weights = torchvision.models.ResNet18_Weights.DEFAULT
dataset = image_folder('data/resnet_images', transform=weights.transforms())
groups = split_dataset(dataset, stratify=True, validation_fraction=0.2, seed=42)
training = groups[0]
validation = groups[1]
model = torchvision.models.resnet18(weights=weights)
model.fc = torch.nn.Linear(model.fc.in_features, len(dataset.classes))
print('Classes:', dataset.classes)
print('Train/validation:', len(training), len(validation))


## Phase 1 — classifier fine-tuning

Freeze the pretrained ResNet18 backbone and optimize only `model.fc` for **10 epochs**. The class split is reproducible and separate from the training set.

**Note:** This example needs at least two labeled images per class. For correlated images from the same subject, use a *group-aware split* instead.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='resnet_finetune_300', assignments=[['head-tune', 'head-tune']], prefix='')
from lispyn.workflow import train
head_run = train(model, training, head='fc', validation=validation, loss_fn=torch.nn.CrossEntropyLoss(), epochs=10, batch_size=16, lr=0.001, seed=42, metrics='accuracy', experiment_dir='runs/resnet/head')
print('Head validation accuracy:', head_run.history['val_accuracy'])


## Phase 2 — deep/full fine-tuning: 300 epochs

Unfreeze **every ResNet18 layer**, create a new optimizer with a smaller learning rate, and train for **300 epochs**. This is full fine-tuning of pretrained weights, **not** training from scratch.

A real 300-epoch run may take hours and require a GPU. Run it only with your own dataset and appropriate resources.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='resnet_finetune_300', assignments=[['full-tune', 'full-tune']], prefix='')
from lispyn.training import trainable
from lispyn.workflow import train
_hy_gensym_lispyn_model_1 = head_run.model
trainable(_hy_gensym_lispyn_model_1)
deep_run = train(_hy_gensym_lispyn_model_1, training, validation=validation, loss_fn=torch.nn.CrossEntropyLoss(), epochs=300, batch_size=16, lr=1e-05, seed=42, metrics='accuracy', experiment_dir='runs/resnet/deep')
print('Full tuning best accuracy:', max(deep_run.history['val_accuracy']))


## Curves and export

Use one Lisp macro to define a consistent Matplotlib style for **both** training curves. The best checkpoints and JSON histories are saved under `runs/resnet/`.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='resnet_finetune_300', assignments=[['defcurve', 'defcurve']], prefix='')
from lispyn.plots import make_plotter
resnet_curve = make_plotter(theme='paper', train_color='#2563EB', val_color='#D97706', xlabel='Epoch', train_label='Train', val_label='Validation', figsize=[9, 4])
resnet_curve(deep_run, metric='loss', title='ResNet18 — 300-epoch loss', ylabel='Cross-entropy loss', save='runs/resnet/deep/loss.png')
resnet_curve(deep_run, metric='accuracy', title='ResNet18 — 300-epoch accuracy', ylabel='Top-1 accuracy', save='runs/resnet/deep/accuracy.png')
print('Saved ResNet18 plots, histories, and best checkpoints.')
